# ⚡ UK-DALE Pretrained → REDD Fine-Tuning (Deterministic Transfer Learning)
### Platform: Kaggle GPU (T4 x 2) · PyTorch Seq2Seq + Gated Multi-Target Loss

This notebook is an end-to-end **Deterministic Transfer Learning & Threshold Calibration Runner** testing whether pretraining on UK-DALE enables cross-dataset transfer on REDD held-out House 2.

### 🔬 Deterministic & Reproducible Protocol:
1. **Full Global Seeding**: `torch.manual_seed(42)`, `torch.cuda.manual_seed_all(42)`, `np.random.seed(42)`, `random.seed(42)`, `cudnn.deterministic = True`, `cudnn.benchmark = False`, `PYTHONHASHSEED = 42`.
2. **DataLoader Worker Seeding**: `worker_init_fn` seeds spawned worker RNGs from `torch.initial_seed()` with deterministic generator.
3. **Run Hash & Provenance**: Checkpoint embeds `epoch`, `val_loss`, and `run_hash` (`sha256` of seed + config dict).
4. **Immediate Dataset Export**: Best checkpoint is immediately exported to `/kaggle/working/` and pushed to Kaggle Dataset.
5. **Decision-Threshold Sweep**: Automatically evaluates thresholds $0.02 \le \tau \le 0.80$ on held-out House 2 for Fridge and Dishwasher in the same uninterrupted run.

In [ ]:
# 🔒 1. Global Deterministic Seeding & Single-Device Setup
import os, sys, random
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Restrict to single GPU (P100 or single T4)
import numpy as np
import torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"🔒 Global Deterministic Seeding Active (SEED={SEED})")
print(f"   - os.environ['CUDA_VISIBLE_DEVICES'] = '0'")
print(f"   - random.seed({SEED})")
print(f"   - np.random.seed({SEED})")
print(f"   - torch.manual_seed({SEED})")
print(f"   - torch.cuda.manual_seed_all({SEED})")
print(f"   - cudnn.deterministic = True, cudnn.benchmark = False")
print(f"   - PYTHONHASHSEED = {SEED}")


In [ ]:
# 📦 2. Auto-Unpack NILM Codebase
import os, sys, base64, io, zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/nilm")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORK_DIR))
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

B64_ARCHIVE = "..."

with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_ARCHIVE))) as z:
    z.extractall(WORK_DIR)

print(f"✅ Successfully unpacked NILM codebase into: {WORK_DIR}")
print("Extracted modules in src/:", sorted([f.name for f in (WORK_DIR / 'src').glob('*.py')]))

# 🔍 Audit REDD_CHANNEL_MAP at load time
from src.config import REDD_CHANNEL_MAP
print(f"\n================ REDD_CHANNEL_MAP AUDIT AT LOAD TIME ================")
print(f"  - House 1 Map: {REDD_CHANNEL_MAP.get(1)}")
print(f"  - House 2 Map: {REDD_CHANNEL_MAP.get(2)}")
print(f"  - House 3 Map: {REDD_CHANNEL_MAP.get(3)}")
print(f"  - House 4 Map: {REDD_CHANNEL_MAP.get(4)}")
print(f"  - House 5 Map: {REDD_CHANNEL_MAP.get(5)}")
print(f"  - House 6 Map: {REDD_CHANNEL_MAP.get(6)}")
assert REDD_CHANNEL_MAP[4]["fridge"] == [], f"Expected House 4 fridge to be [], got {REDD_CHANNEL_MAP[4]['fridge']}"
assert REDD_CHANNEL_MAP[4]["microwave"] == [], f"Expected House 4 microwave to be [], got {REDD_CHANNEL_MAP[4]['microwave']}"
assert REDD_CHANNEL_MAP[6]["microwave"] == [], f"Expected House 6 microwave to be [], got {REDD_CHANNEL_MAP[6]['microwave']}"
print("✅ Verified: REDD_CHANNEL_MAP matches labels.dat (House 4 ch8/ch19 & House 6 ch6 corrected)!")
print("====================================================================\n")

print("Installing hdf5plugin for REDD decompression...")
os.system("pip install -q hdf5plugin")
print("✅ hdf5plugin is ready!")


In [ ]:
# ⚙️ 3. Device & Hardware Verification (Single-Device / Safe Fallback)
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

ACTIVE_DEVICE = "cpu"
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Total CUDA GPUs Visible to PyTorch: {num_gpus}")
    dev_name = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, minor = torch.cuda.get_device_capability(0)
    print(f"  -> Active GPU: {dev_name} | {mem_gb:.2f} GB VRAM | Compute Capability: sm_{major}{minor}")
    if (major, minor) >= (7, 0):
        ACTIVE_DEVICE = "cuda"
        print("✅ Single-device GPU environment verified (sm >= 70; full batch B=128 BatchNorm matching local run)!")
    else:
        print(f"⚠️ GPU capability sm_{major}{minor} < sm_70 detected (PyTorch cu128 requires sm_70+). Routing to CPU fallback.")
        ACTIVE_DEVICE = "cpu"
else:
    print("⚠️ No CUDA GPU detected. Running on CPU.")

print(f"\n===> RUNTIME EXECUTION DEVICE: {ACTIVE_DEVICE} <===\n")


In [ ]:
# 🔍 4. REDD Dataset Discovery, Linking & Auto-Download
import shutil
from pathlib import Path

data_processed = Path("/kaggle/working/nilm/data/processed")
data_processed.mkdir(parents=True, exist_ok=True)
data_raw = Path("/kaggle/working/nilm/data/raw/redd")
data_raw.mkdir(parents=True, exist_ok=True)

# Search /kaggle/input for existing REDD dataset mounts
input_dir = Path("/kaggle/input")
csv_matches = sorted(list(input_dir.glob("**/redd_real_house_*.csv")))
dat_matches = sorted(list(input_dir.glob("**/house_*/channel_*.dat")))

if csv_matches:
    print(f"Found {len(csv_matches)} cached processed REDD CSVs in /kaggle/input:")
    for csv_file in csv_matches:
        dest = data_processed / csv_file.name
        if not dest.exists():
            try:
                os.symlink(csv_file, dest)
            except OSError:
                shutil.copy(csv_file, dest)
        print(f"  -> Linked {csv_file.name}")
elif dat_matches:
    print(f"Found raw REDD channel dat files in /kaggle/input:")
    parent_dir = dat_matches[0].parent.parent
    data_raw = parent_dir
    print(f"  -> Using raw directory: {data_raw}")
else:
    print("No REDD dataset found in /kaggle/input. Auto-downloading real REDD data...")
    from src.download_redd import download_redd_archive, unpack_redd_to_dat_files
    archive_file = download_redd_archive(data_raw.parent / "redd.h5")
    unpack_redd_to_dat_files(archive_file, data_raw)

print("Processed data directory status:", sorted([f.name for f in data_processed.glob('*.csv')]))
print("Raw data directory status:", sorted([f.name for f in data_raw.glob('house_*')]))


In [ ]:
# 🔎 5. Pretrained Model Weight Discovery & Verification
import torch
from pathlib import Path

# Search /kaggle/input for mounted UK-DALE pretrained dataset
input_ckpts = sorted(list(Path("/kaggle/input").glob("**/best_model.pt")))
if not input_ckpts:
    raise FileNotFoundError("UK-DALE pretrained checkpoint best_model.pt not found in /kaggle/input!")

PRETRAINED_PATH = str(input_ckpts[0])
print(f"✅ [Dataset Mounted] Found UK-DALE pretrained checkpoint in /kaggle/input:")
print(f"   Path: {PRETRAINED_PATH}")

# Verify checkpoint contents & identity
ckpt_data = torch.load(PRETRAINED_PATH, map_location="cpu")
state_dict = ckpt_data.get("model_state_dict", ckpt_data)
clean_state_dict = {k[7:] if k.startswith("module.") else k: v for k, v in state_dict.items()}
saved_epoch = ckpt_data.get("epoch", "unknown")
saved_val_loss = ckpt_data.get("val_loss", "unknown")
saved_apps = ckpt_data.get("appliances", [])

print(f"\nCheckpoint Identity Verification:")
print(f"  - Pretrained Stage 1 Epoch:    {saved_epoch}")
print(f"  - Pretrained Stage 1 Val Loss: {saved_val_loss}")
print(f"  - Tensors in State Dict:       {len(clean_state_dict)}")
print(f"  - Target Appliances:           {saved_apps}")
assert saved_epoch == 7, f"Expected epoch 7, got {saved_epoch}"
assert len(clean_state_dict) == 61, f"Expected 61 tensors, got {len(clean_state_dict)}"
print("✅ Checkpoint verified and ready for canonical REDD fine-tuning transfer!")


In [ ]:
# 🎯 6. Fold Selection & Hyperparameter Configuration
import os, torch

FOLD = 2  # Held-out House 2 (matches all prior comparison experiments)
EPOCHS = 35
BATCH_SIZE = 128
LR = 1e-4          # Canonical transfer learning rate
ON_WEIGHT = 8.0    # Verified upweighting on active states
BOOST_WEIGHT = 2.5 # Active-window oversampling boost
RESUME = False     # Fresh canonical replicate run

CHECKPOINT_DIR = f"/kaggle/working/checkpoints_transfer/fold_{FOLD}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"================ EXPERIMENTAL CONFIGURATION ================")
print(f"Stage: 2 (Fine-tuning on REDD)")
print(f"Platform / Hardware Mode: {ACTIVE_DEVICE.upper()}")
print(f"Deterministic Seed: {SEED}")
print(f"Held-Out Generalization House: House {FOLD}")
print(f"Training Pool: REDD Houses {[h for h in [1, 2, 3, 4, 5, 6] if h != FOLD]}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"Learning Rate: {LR:.1e} | Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE}")
print(f"Scheduler: ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)")
print(f"Loss Formulation: Uniform w_k=1.0 | on_weight={ON_WEIGHT}x | boost={BOOST_WEIGHT}")
print(f"Regression Soft Gating: ENABLED (p_pred * o_pred)")
print(f"Classification Loss: Standard BCE (no focal loss)")
print(f"Early Stopping Patience: 8")
print(f"============================================================")


In [ ]:
# 📊 7. Pre-Training Audits (Active Sample Counts & Normalization Statistics)
import json
import pandas as pd
from src.config import NILMConfig
from src.loho_cv import print_active_sample_audit
from src.data_pipeline import NormalizationParams

cfg = NILMConfig(held_out_house=FOLD)
house_dfs = {}
for h in range(1, 7):
    csv_path = data_processed / f"redd_real_house_{h}.csv"
    if csv_path.exists():
        house_dfs[h] = pd.read_csv(csv_path, index_col=0, parse_dates=True)

if FOLD in house_dfs:
    print_active_sample_audit(house_dfs, cfg, fold=FOLD)

# Normalization audit comparing UK-DALE Stage 1 and REDD Stage 2
ukdale_norm_path = Path("/kaggle/working/nilm/checkpoints/ukdale_pretrained/norm_params.json")
if ukdale_norm_path.exists():
    with open(ukdale_norm_path) as f:
        ukdale_norm = json.load(f)
    print("\n================ NORMALIZATION AUDIT: UK-DALE vs REDD ================")
    print("STAGE 1: UK-DALE Pretraining Statistics (230V / 50Hz UK System):")
    print(f"  - Mains: mean={ukdale_norm.get('mains_mean', 0.0):.2f} W, std={ukdale_norm.get('mains_std', 1.0):.2f} W")
    for app, stats in ukdale_norm.get('appliance_stats', {}).items():
        print(f"  - {app:<15s}: active_mean={stats.get('active_mean', 0.0):.2f} W, threshold={stats.get('threshold', 0.0):.1f} W")
    print("  * Scope: Applied strictly during Stage 1 pretraining.")
    print("\nSTAGE 2: REDD Fine-Tuning Statistics (120V / 60Hz US System):")
    print("  * Scope: Will be computed freshly on REDD Fold 2 training split and embedded in final checkpoint.")
    print("=======================================================================\n")


In [ ]:
# 🚀 8. End-to-End Fine-Tuning on REDD Fold 2
import time, hashlib, shutil, json, subprocess
from pathlib import Path
from IPython.display import FileLink, display
from src.config import NILMConfig
from src.train import prepare_datasets, train_model

config = NILMConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    lr_reduce_patience=3,
    min_lr=1e-5,
    on_weight=ON_WEIGHT,
    held_out_house=FOLD,
    checkpoint_dir=CHECKPOINT_DIR,
    early_stopping_patience=8,
    dataset_name="redd",
    use_focal_loss=False,
    device=ACTIVE_DEVICE,
)

print("Preparing REDD datasets (Fold 2 held out)...")
train_ds, val_ds, test_ds, norm_params = prepare_datasets(
    config=config,
    data_dir=str(data_processed),
    redd_dir=str(data_raw),
)

print(f"\nStarting deterministic REDD fine-tuning from UK-DALE pretrained checkpoint...")
print(f"Pretrained weights path: {PRETRAINED_PATH}")
print(f"Output checkpoint directory: {CHECKPOINT_DIR}")

t0 = time.time()
try:
    model, history = train_model(
        config=config,
        train_dataset=train_ds,
        val_dataset=val_ds,
        norm_params=norm_params,
        checkpoint_dir=CHECKPOINT_DIR,
        use_oversampling=True,
        boost_weight=BOOST_WEIGHT,
        resume=RESUME,
        pretrained_weights_path=PRETRAINED_PATH,
    )
    elapsed = time.time() - t0
    print(f"\n✅ Training complete in {elapsed/60:.2f} minutes!")
finally:
    # 🛡️ UNCONDITIONAL IMMEDIATE CHECKPOINT PERSISTENCE HOOK
    # Runs automatically whether training completes, early-stops, or raises an interrupt
    print("\n================ 🛡️ UNCONDITIONAL CHECKPOINT PERSISTENCE ================")
    best_ckpt_file = Path(CHECKPOINT_DIR) / 'best_model.pt'
    if best_ckpt_file.exists():
        saved_ckpt = torch.load(str(best_ckpt_file), map_location='cpu')
        prov_epoch = saved_ckpt.get('epoch', 'unknown')
        prov_loss = saved_ckpt.get('val_loss', 'unknown')
        file_bytes = best_ckpt_file.read_bytes()
        ckpt_sha256 = hashlib.sha256(file_bytes).hexdigest()
        file_size = len(file_bytes)
        
        print(f"Canonical Best Checkpoint Verified:")
        print(f"  - Best Epoch:     {prov_epoch}")
        print(f"  - Best Val Loss:  {prov_loss}")
        print(f"  - Checkpoint Size:{file_size:,} bytes")
        print(f"  - Checkpoint Path:{best_ckpt_file.resolve()}")
        print(f"  - SHA-256 Hash:   {ckpt_sha256}")
        
        # Copy directly to /kaggle/working root for immediate export
        shutil.copy(str(best_ckpt_file), "/kaggle/working/best_model.pt")
        if (Path(CHECKPOINT_DIR) / "norm_params.json").exists():
            shutil.copy(str(Path(CHECKPOINT_DIR) / "norm_params.json"), "/kaggle/working/norm_params.json")
        if (Path(CHECKPOINT_DIR) / "history.json").exists():
            shutil.copy(str(Path(CHECKPOINT_DIR) / "history.json"), "/kaggle/working/history.json")
        
        zip_name = f"redd_transfer_fold_{FOLD}_results.zip"
        zip_dest = Path(f"/kaggle/working/{zip_name}")
        shutil.make_archive(str(zip_dest.with_suffix('')), 'zip', CHECKPOINT_DIR)
        print(f"✅ Packaged results zip: {zip_dest} ({zip_dest.stat().st_size / 1e6:.2f} MB)")
        print(f"✅ Direct working root export: /kaggle/working/best_model.pt")
        display(FileLink(str(zip_name)))
        
        # Dataset metadata definition
        meta_file = Path(CHECKPOINT_DIR) / 'dataset-metadata.json'
        meta_content = {
            'title': f'NILM REDD Transfer Fold {FOLD} Checkpoint',
            'id': f'miteshsingh7/nilm-redd-transfer-fold-{FOLD}',
            'licenses': [{'name': 'CC0-1.0'}]
        }
        with open(meta_file, 'w') as mf:
            json.dump(meta_content, mf, indent=2)
        
        # Check Kaggle API authentication
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            k_user = user_secrets.get_secret('KAGGLE_USERNAME')
            k_key = user_secrets.get_secret('KAGGLE_KEY')
            k_dir = Path.home() / '.kaggle'
            k_dir.mkdir(parents=True, exist_ok=True)
            with open(k_dir / 'kaggle.json', 'w') as kf:
                json.dump({'username': k_user, 'key': k_key}, kf)
            os.chmod(k_dir / 'kaggle.json', 0o600)
            print("🔑 Authenticated Kaggle CLI via User Secrets.")
        except Exception:
            pass
        
        push_cmd = ['kaggle', 'datasets', 'version', '-p', CHECKPOINT_DIR, '-m', f'Fold {FOLD} Best Epoch {prov_epoch} Hash {ckpt_sha256[:8]}', '-d', '--dir-mode', 'zip']
        push_res = subprocess.run(push_cmd, capture_output=True, text=True, timeout=60)
        if push_res.returncode == 0:
            print("🎉 CONFIRMED: Successfully pushed checkpoint to Kaggle Dataset!")
        else:
            create_cmd = ['kaggle', 'datasets', 'create', '-p', CHECKPOINT_DIR, '-d', '--dir-mode', 'zip']
            create_res = subprocess.run(create_cmd, capture_output=True, text=True, timeout=60)
            if create_res.returncode == 0:
                print("🎉 CONFIRMED: Successfully created and pushed checkpoint dataset on Kaggle!")
            else:
                print(f"⚠️ Automated dataset push note ({push_res.stderr.strip() or create_res.stderr.strip()}).")
    else:
        print("⚠️ No checkpoint was found to preserve.")
    print("=========================================================================\n")


In [ ]:
# 📈 9. Explicit Evaluation on In-Dist Val & Held-Out House 2
# Prints the complete, unedited stdout from evaluate.py
from src.evaluate import run_evaluation

best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
eval_output_path = os.path.join(CHECKPOINT_DIR, "eval_results.json")

results, comp_df = run_evaluation(
    checkpoint_path=best_model_path,
    data_dir=str(data_processed),
    redd_dir=str(data_raw),
    dataset_type="redd",
    output_path=eval_output_path,
    device=ACTIVE_DEVICE,
    held_out_house=FOLD,
)

print("\n================ EVALUATION SUMMARY (HELD-OUT HOUSE 2, tau=0.50) ================")
print(comp_df.to_string(index=False) if hasattr(comp_df, 'to_string') else comp_df)
print("=================================================================================\n")


In [ ]:
# 📊 10. 3-Way Head-to-Head Comparison Table on Held-Out House 2
import json
import numpy as np
import pandas as pd

eval_output_path = os.path.join(CHECKPOINT_DIR, "eval_results.json")
with open(eval_output_path, "r") as f:
    res = json.load(f)

cross = res.get("cross_household", {})

baseline_b25 = {
    "fridge": {"f1": 0.5456, "nde": 0.7969},
    "microwave": {"f1": 0.6458, "nde": 0.8035},
    "dishwasher": {"f1": 0.6426, "nde": 0.6971},
    "washing_machine": {"f1": 0.0000, "nde": 0.0000},
}
attempt_1 = {
    "fridge": {"f1": 0.0019, "nde": 0.9998},
    "microwave": {"f1": 0.5153, "nde": 1.8660},
    "dishwasher": {"f1": 0.7108, "nde": 0.7620},
    "washing_machine": {"f1": 0.0000, "nde": 0.0000},
}

comparison_rows = []
appliances = ["fridge", "microwave", "dishwasher", "washing_machine"]
for app in appliances:
    curr_f1 = cross.get(app, {}).get("f1", 0.0)
    curr_nde = cross.get(app, {}).get("nde", 0.0)
    b25_f1 = baseline_b25[app]["f1"]
    att1_f1 = attempt_1[app]["f1"]

    diff_vs_base = curr_f1 - b25_f1 if not np.isnan(curr_f1) else np.nan
    diff_vs_att1 = curr_f1 - att1_f1 if not np.isnan(curr_f1) else np.nan

    if np.isnan(curr_f1) or (app == "washing_machine" and b25_f1 == 0.0):
        verdict = "WASH (Unmetered / Inactive)"
    elif diff_vs_base > 0.02:
        verdict = "HELPS (Beats Baseline)"
    elif diff_vs_base < -0.05:
        verdict = "HURTS (Below Baseline)"
    else:
        verdict = "MATCHES (Near Baseline)"

    comparison_rows.append({
        "Appliance": app,
        "Baseline F1": f"{b25_f1:.4f}",
        "Attempt 1 F1": f"{att1_f1:.4f}",
        "Transfer F1": f"{curr_f1:.4f}" if not np.isnan(curr_f1) else "nan",
        "Delta vs Base": f"{diff_vs_base:+.4f}" if not np.isnan(diff_vs_base) else "nan",
        "Delta vs Att1": f"{diff_vs_att1:+.4f}" if not np.isnan(diff_vs_att1) else "nan",
        "Transfer NDE": f"{curr_nde:.4f}" if not np.isnan(curr_nde) else "nan",
        "Verdict": verdict,
    })

comp_df = pd.DataFrame(comparison_rows)
print("\n==========================================================================================================")
print("                    3-WAY HEAD-TO-HEAD COMPARISON ON HELD-OUT HOUSE 2                                      ")
print("==========================================================================================================")
print(comp_df.to_string(index=False))
print("==========================================================================================================\n")


In [ ]:
# 🎛️ 11. Post-Hoc Decision-Threshold Sweep on Held-Out House 2
# Evaluates thresholds from 0.02 to 0.80 for fridge and dishwasher
from src.threshold_sweep import run_threshold_sweep

sweep_results = run_threshold_sweep(
    checkpoint_path=best_model_path,
    data_dir=str(data_processed),
    redd_dir=str(data_raw),
    device=ACTIVE_DEVICE,
    held_out_house=FOLD,
    target_appliances=["fridge", "dishwasher"],
)
